# RiskModels API — Quickstart Notebook

This notebook walks through the key use cases for the [RiskModels](https://riskmodels.net) API:

**Risk & Hedging**
1. **Hedge a single stock** — get L1/L2/L3 hedge ratios for any ticker
2. **Hedge a portfolio** — compute weighted portfolio-level hedge ratios across positions
3. **Build a factor risk table** — decompose returns into market, sector, subsector, and residual components

**Account & Billing**

4. **Balance & status** — check your current balance, rate limits, and account health
5. **Invoice history** — view spend by period and usage by API capability
6. **Ticker metrics snapshot** — latest volatility, Sharpe, hedge ratios, and sector codes for any ticker

---

### Setup
1. Paste your API key into the cell below (replace `PASTE_YOUR_KEY_HERE`)
2. Run all cells top-to-bottom (`Runtime → Run all`)

> **Tip:** Use `File → Save a copy in Drive` to keep a private copy with your key.


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
API_KEY  = "PASTE_YOUR_KEY_HERE"   # <-- paste your RiskModels API key here
BASE_URL = "https://riskmodels.net/api"
HEADERS  = {"Authorization": f"Bearer {API_KEY}"}

import requests
import pandas as pd

# Quick sanity check
if API_KEY == "PASTE_YOUR_KEY_HERE":
    raise ValueError("Please paste your API key above before running.")

print("Config ready. Key prefix:", API_KEY[:16] + "...")


---
## Use Case 1 — How do I hedge a stock?

The `/ticker-returns` endpoint returns a **full daily time series** of gross returns and rolling hedge ratios for any ticker going back up to 15 years.

| Field | Meaning |
|---|---|
| `stock_return` | Daily gross return of the stock |
| `l1_hedge` | Market hedge ratio vs SPY (L1 Model, rolling) |
| `l3_market_hr` | Market leg ratio of L3 model (rolling) |
| `l3_sector_hr` | Sector leg ratio of L3 model (rolling) |

The **latest row** gives the current hedge ratio to use for a live trade.
For the full 6-component breakdown (separate SPY / sector / subsector notionals), use the `/metrics/{ticker}` endpoint.


In [ ]:
# ── Use Case 1: Hedge a single stock ───────────────────────────────────────────
ticker = "NVDA"   # change to any ticker, e.g. "AAPL", "TSLA", "MSFT"

resp = requests.get(
    f"{BASE_URL}/ticker-returns",
    headers=HEADERS,
    params={"ticker": ticker, "years": 1}
)
resp.raise_for_status()
body = resp.json()

# Map response into a DataFrame
df = pd.DataFrame(body["data"])
df.rename(columns={
    "stock": "stock_return",
    "l1_market_hr":    "l1_hedge",
    "l3_market_hr":    "l3_mkt_leg",
    "l3_sector_hr":    "l3_sec_leg",
    "l3_subsector_hr": "l3_sub_leg",
}, inplace=True)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

meta   = body["meta"]
latest = df.iloc[-1]

# ── Latest hedge ratios — transposed for readability ──────────────────────────
snapshot = pd.DataFrame({
    "Value": {
        "market_etf":               meta["market_etf"],
        "sector_etf":               meta["sector_etf"],
        "subsector_etf":            meta["subsector_etf"],
        "L1 hedge (market only)":   round(latest.l1_hedge, 4),
        "L3 Market Leg":           round(latest.l3_mkt_leg, 4),
        "L3 Sector Leg":           round(latest.l3_sec_leg, 4),
        "L3 Subsector Leg":        round(latest.l3_sub_leg, 4),
    }
})
print(f"Latest hedge ratios — {ticker}")
display(snapshot)

# ── Recent history ─────────────────────────────────────────────────────────────
print(f"\nMost recent 10 trading days:")
df[["date", "stock_return", "l1_hedge", "l3_mkt_leg", "l3_sec_leg", "l3_sub_leg"]].tail(10)


---
## Use Case 2 — How do I hedge a portfolio?

The `/batch/analyze` endpoint fetches the **full 6-component hedge breakdown** for multiple tickers in one call (25% cheaper than individual calls).

| Field | Meaning |
|---|---|
| `l1_market` | SPY ratio for L1 hedge — 1 trade |
| `l2_market` | SPY ratio for L2 hedge (reduced vs L1) |
| `l2_sector` | Sector ETF ratio for L2 |
| `l3_market` | SPY ratio for L3 hedge (further reduced) |
| `l3_sector` | Sector ETF ratio for L3 |
| `l3_subsector` | Subsector ETF ratio for L3 (can be positive = long) |

Multiply each ratio by position size to get the notional hedge amount per leg.


In [ ]:
# ── Use Case 2: Hedge a portfolio ──────────────────────────────────────────────
# Define portfolio: ticker -> weight (weights should sum to 1.0)
portfolio = {
    "AAPL":  0.25,
    "MSFT":  0.20,
    "NVDA":  0.20,
    "GOOGL": 0.15,
    "AMZN":  0.10,
    "JPM":   0.10,
}

resp = requests.post(
    f"{BASE_URL}/batch/analyze",
    headers=HEADERS,
    json={
        "tickers": list(portfolio.keys()),
        "metrics": ["hedge_ratios"],
        "years": 1,
    }
)
resp.raise_for_status()
results = resp.json()["results"]

# ── Per-position breakdown ─────────────────────────────────────────────────────
rows = []
for ticker, weight in portfolio.items():
    r  = results.get(ticker, {})
    hr = r.get("hedge_ratios") or {}   # null-safe: API returns null if ticker missing
    rows.append({
        "ticker":       ticker,
        "weight":       weight,
        "status":       r.get("status", "error"),
        "l1_market_hr": hr.get("l1_market"),
        "l2_market_hr": hr.get("l2_market"),
        "l2_sector_hr": hr.get("l2_sector"),
        "l3_market_hr": hr.get("l3_market"),
        "l3_sector_hr": hr.get("l3_sector"),
        "l3_sub_hr":    hr.get("l3_subsector"),
    })

df_port = pd.DataFrame(rows).set_index("ticker")

# ── Weighted portfolio-level hedge ratios ──────────────────────────────────────
for col in ["l1_market_hr", "l2_market_hr", "l3_market_hr"]:
    df_port[f"w_{col}"] = df_port["weight"] * df_port[col]

port_summary = pd.DataFrame({
    "Value": {
        "L1 market hedge (wtd)": round(df_port["w_l1_market_hr"].sum(), 4),
        "L2 market hedge (wtd)": round(df_port["w_l2_market_hr"].sum(), 4),
        "L3 market hedge (wtd)": round(df_port["w_l3_market_hr"].sum(), 4),
    }
})
print("Portfolio-level hedge ratios (weighted average):")
display(port_summary)

# ── Per-position table ─────────────────────────────────────────────────────────
print("\nPer-position breakdown:")
display(df_port[["weight", "status", "l1_market_hr", "l2_market_hr",
                  "l2_sector_hr", "l3_market_hr", "l3_sector_hr", "l3_sub_hr"]])


---
## Use Case 3 — Build a risk table with factor components

The `/l3-decomposition` endpoint returns monthly explained returns decomposed into four factors:

| Column | Meaning |
|---|---|
| `market_%` | Return explained by the broad market (SPY) |
| `sector_%` | Return explained by the sector ETF |
| `subsector_%` | Return explained by the subsector ETF |
| `residual_%` | Idiosyncratic / stock-specific return |
| `total_%` | Sum of all four components |

This is the building block for a full factor risk attribution table across a portfolio.


In [ ]:
# ── Use Case 3: Factor risk attribution table ──────────────────────────────────
ticker = "NVDA"   # change to any ticker

resp = requests.get(
    f"{BASE_URL}/l3-decomposition",
    headers=HEADERS,
    params={"ticker": ticker, "market_factor_etf": "SPY"}
)
resp.raise_for_status()
body = resp.json()

# Map columnar response into a tidy DataFrame
df_risk = pd.DataFrame({
    "date":         pd.to_datetime(body["dates"]),
    "market_er":    body["l3_market_er"],
    "sector_er":    body["l3_sector_er"],
    "subsector_er": body["l3_subsector_er"],
    "residual_er":  body["l3_residual_er"],
})
df_risk = df_risk.dropna().sort_values("date").reset_index(drop=True)

# Total return = sum of all factor components
df_risk["total_return"] = df_risk[["market_er", "sector_er",
                                    "subsector_er", "residual_er"]].sum(axis=1)

# Convert to percentages for readability
pct_cols = ["market_er", "sector_er", "subsector_er", "residual_er", "total_return"]
df_risk[pct_cols] = (df_risk[pct_cols] * 100).round(3)
df_risk.rename(columns={c: c.replace("_er", "_%").replace("_return", "_%")
                         for c in pct_cols}, inplace=True)

print(f"Monthly factor risk attribution for {ticker} (most recent 12 months)")
print(f"Market ETF: {body['market_factor_etf']}  |  Universe: {body['universe']}")
print()

df_risk.tail(12)


### Bonus: Factor risk table across a full portfolio

Loop the same call across multiple tickers and stack into one table.


In [ ]:
# ── Bonus: Portfolio-level factor risk table ───────────────────────────────────
tickers = ["AAPL", "MSFT", "NVDA", "GOOGL"]  # add any tickers here

all_rows = []
for t in tickers:
    r = requests.get(
        f"{BASE_URL}/l3-decomposition",
        headers=HEADERS,
        params={"ticker": t, "market_factor_etf": "SPY"}
    )
    if r.status_code != 200:
        print(f"Warning: {t} returned {r.status_code}")
        continue
    b = r.json()
    tmp = pd.DataFrame({
        "date":         pd.to_datetime(b["dates"]),
        "market_er":    b["l3_market_er"],
        "sector_er":    b["l3_sector_er"],
        "subsector_er": b["l3_subsector_er"],
        "residual_er":  b["l3_residual_er"],
    })
    tmp["ticker"] = t
    all_rows.append(tmp)

df_all = pd.concat(all_rows, ignore_index=True).dropna()

# Summarise: mean monthly factor attribution per ticker
summary = (
    df_all
    .groupby("ticker")[["market_er", "sector_er", "subsector_er", "residual_er"]]
    .mean()
    .multiply(100)
    .round(3)
)
summary.columns = ["market_%", "sector_%", "subsector_%", "residual_%"]
summary["total_%"] = summary.sum(axis=1).round(3)

print("Average monthly factor attribution by ticker (in %):")
summary


---
## Use Case 4 — Precision Hedge Chart: stock vs. factor cumulative returns

The time series from `/ticker-returns` lets you visualise exactly how much of a stock's return is explained by each hedge layer — market, sector, and subsector.

The chart shows **cumulative compound returns** for:
- **Stock** — the raw position (e.g. NVDA)
- **Market** — what a pure SPY hedge would have returned over the same period (`l3_market_er` column)
- **Sector** — the sector ETF component (`l3_sector_er` column)
- **Subsector** — the subsector ETF component (`l3_subsector_er` column)

The gap between the stock line and the hedge lines = the **residual / idiosyncratic return** that ETF hedges cannot capture.

> **Cost:** `$0.005` per `/ticker-returns` call, regardless of how many years you pull. Pulling 15 years of daily data costs the same as pulling 1 day.


In [ ]:
# ── Use Case 4: Precision Hedge Chart ──────────────────────────────────────────
# pip install matplotlib   (pre-installed in Colab)

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

TICKER = "NVDA"   # change to any ticker
YEARS  = 3        # 1, 3, 5, or 15

# ── Fetch time series ──────────────────────────────────────────────────────────
resp = requests.get(
    f"{BASE_URL}/ticker-returns",
    headers=HEADERS,
    params={"ticker": TICKER, "years": YEARS},
)
resp.raise_for_status()
df = pd.DataFrame(resp.json()["data"])
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# ── Compute cumulative compound returns (geometric) ────────────────────────────
# Formula: cum[t] = (1 + cum[t-1]) * (1 + daily[t]) - 1
# All series start at 0% on day 0.
def cumulative(series):
    cum = np.zeros(len(series))
    for i, r in enumerate(series):
        if i == 0:
            cum[i] = r
        else:
            cum[i] = (1 + cum[i - 1]) * (1 + r) - 1
    return cum * 100   # convert to %

df["cum_stock"]     = cumulative(df["stock"])
df["cum_market"]    = cumulative(df["l3_market_er"])
df["cum_sector"]    = cumulative(df["l3_sector_er"])
df["cum_subsector"] = cumulative(df["l3_subsector_er"])

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#0d0d0d")
ax.set_facecolor("#0d0d0d")

colors = {
    "cum_stock":     "#60a5fa",   # blue  — stock
    "cum_market":    "#6366f1",   # indigo — market (SPY)
    "cum_sector":    "#34d399",   # green — sector ETF
    "cum_subsector": "#9ca3af",   # grey  — subsector ETF
}
labels = {
    "cum_stock":     TICKER,
    "cum_market":    "Market (L1)",
    "cum_sector":    "Sector (L2)",
    "cum_subsector": "Subsector (L3)",
}

for col, color in colors.items():
    ax.plot(df["date"], df[col], color=color, linewidth=1.4, label=labels[col])

ax.axhline(0, color="#444", linewidth=0.8, linestyle="--")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.tick_params(colors="#aaa", labelsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#333")
ax.set_xlabel("Date", color="#aaa", fontsize=9)
ax.set_ylabel("Cumulative Return", color="#aaa", fontsize=9)
ax.set_title(
    f"Your Precision Hedge Recipe — {TICKER}  ({YEARS}y)",
    color="white", fontsize=12, pad=10
)
ax.legend(
    frameon=False, labelcolor="#ccc", fontsize=9,
    loc="upper left", title="Series", title_fontsize=8,
)
ax.grid(axis="y", color="#222", linewidth=0.6)
plt.tight_layout()
plt.show()

# ── Latest values ──────────────────────────────────────────────────────────────
latest = df.iloc[-1]
summary = pd.DataFrame({
    "Value": {
        f"{TICKER} total return":    f"{latest.cum_stock:.1f}%",
        "Market factor return":      f"{latest.cum_market:.1f}%",
        "Sector factor return":      f"{latest.cum_sector:.1f}%",
        "Subsector factor return":   f"{latest.cum_subsector:.1f}%",
        "Residual (unhedgeable)":    f"{latest.cum_stock - latest.cum_subsector:.1f}%",
    }
})
print(f"Cumulative returns over {YEARS}y — as of {latest.date.date()}")
display(summary)


---
## Next Steps

- **Documentation:** [riskmodels.net/docs/api](https://riskmodels.net/docs/api)
- **Top up balance:** `POST https://riskmodels.net/api/billing/top-up`
- **Questions?** Email [api-support@riskmodels.net](mailto:api-support@riskmodels.net)

Happy modelling!


---
## Bonus: AI Risk Analyst — GPT-4o with live factor data

Combine the RiskModels API with an LLM to build an AI that can answer natural-language questions about your portfolio.

The pattern is simple:
1. Fetch live hedge ratios and risk metrics from the RiskModels API
2. Inject the data as structured context into a system prompt
3. Ask any hedging or risk question — the model reasons over real numbers

> **Requires:** `pip install openai` and an [OpenAI API key](https://platform.openai.com/api-keys)


In [ ]:
# ── AI Risk Analyst — GPT-4o + RiskModels live data ────────────────────────────
# pip install openai   (run once if not already installed)

OPENAI_API_KEY = "PASTE_YOUR_OPENAI_KEY_HERE"   # <-- your OpenAI key

# ── Step 1: Fetch live risk metrics for the portfolio ──────────────────────────
portfolio = {
    "AAPL": 0.25,
    "MSFT": 0.20,
    "NVDA": 0.20,
    "GOOGL": 0.15,
    "AMZN": 0.10,
    "JPM":  0.10,
}

metrics_rows = []
for t in portfolio:
    r = requests.get(f"{BASE_URL}/metrics/{t}", headers=HEADERS)
    if r.status_code == 200:
        m = r.json()
        metrics_rows.append({
            "ticker":          t,
            "weight_%":        round(portfolio[t] * 100, 1),
            "close":           m.get("close_price"),
            "vol_ann_%":       round((m.get("volatility") or 0) * 100, 1),
            "sharpe":          round(m.get("sharpe_ratio") or 0, 3),
            "l1_market_hr":    round(m.get("l1_market_hr") or 0, 4),
            "l2_market_hr":    round(m.get("l2_market_hr") or 0, 4),
            "l2_sector_hr":    round(m.get("l2_sector_hr") or 0, 4),
            "l3_market_hr":    round(m.get("l3_market_hr") or 0, 4),
            "l3_sector_hr":    round(m.get("l3_sector_hr") or 0, 4),
            "l3_subsector_hr": round(m.get("l3_subsector_hr") or 0, 4),
            "l1_market_er":    round(m.get("l1_market_er") or 0, 4),
            "l3_residual_er":  round(m.get("l3_residual_er") or 0, 4),
        })

df_ai = pd.DataFrame(metrics_rows).set_index("ticker")

# ── Step 2: Render the data as a compact text table for the prompt ─────────────
risk_table = df_ai.to_string()

system_prompt = f"""You are an institutional equity risk analyst with expertise in factor models.
You have access to live ERM3 factor data for a portfolio. Use ONLY the numbers provided.

ERM3 Hedge Ratio Guide:
- l1_market_hr: SPY ratio for L1 hedge (market-only, 1 trade)
- l2_market_hr / l2_sector_hr: SPY + sector ETF ratios for L2 hedge (2 trades)
- l3_market_hr / l3_sector_hr / l3_subsector_hr: all three ETFs for L3 hedge (3 trades)
- l1_market_er: fraction of risk explained by market factor (0–1)
- l3_residual_er: idiosyncratic risk fraction — cannot be hedged with ETFs

LIVE PORTFOLIO DATA:
{risk_table}

Answer concisely and always cite specific numbers from the table above."""

# ── Step 3: Ask a question ──────────────────────────────────────────────────────
question = "Which position has the most market risk? What hedge trades should I place to reduce it at L2?"

# ── Step 4: Call GPT-4o ────────────────────────────────────────────────────────
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question},
    ],
    temperature=0.2,
)

print(f"Q: {question}\n")
print(response.choices[0].message.content)


---
## Account — Balance & Status

Check your current balance, account status, and rate-limit settings in one call.


In [ ]:
# ── Account balance & status ────────────────────────────────────────────────────
resp = requests.get(
    f"{BASE_URL}/balance",
    headers=HEADERS,
    params={"include_usage": "true"}
)
resp.raise_for_status()
b = resp.json()

print(f"Account:  {b['email']}")
print(f"Balance:  ${b['balance_usd']:.4f} {b['currency']}")
print(f"Type:     {b['account_type']}")
print(f"Status:   {b['status']['account']}  |  billing: {b['status']['billing']}")
print(f"Can request: {b['status']['can_make_requests']}")
print()
print("Rate limits:")
for k, v in b["limits"].items():
    print(f"  {k}: {v}")

# Flatten into a one-row summary DataFrame
df_balance = pd.DataFrame([{
    "email":          b["email"],
    "balance_usd":    b["balance_usd"],
    "account_type":   b["account_type"],
    "billing_status": b["status"]["billing"],
    "rate_limit_rpm": b["limits"]["rate_limit_per_minute"],
    "daily_limit":    b["limits"]["daily_request_limit"],
    "last_updated":   b["last_updated"],
}])
df_balance


---
## Billing — Invoice History & Spend Summary

`/api/invoices` returns a list of past charges plus a period summary broken down by API capability.


In [ ]:
# ── Invoice history & spend summary ────────────────────────────────────────────
resp = requests.get(
    f"{BASE_URL}/invoices",
    headers=HEADERS,
    params={"limit": 20, "period": "month"}
)
resp.raise_for_status()
inv = resp.json()

summary = inv["summary"]
print(f"Period:          {summary['period']}")
print(f"Total invoices:  {summary['total_invoices']}")
print(f"Total spent:     ${summary['total_spent_usd']:.4f}")
print(f"Total requests:  {summary['total_requests']}")
print(f"Current period:  ${summary['current_period_cost_usd']:.4f}")
print()

# Invoice list as DataFrame
df_inv = pd.DataFrame(inv["invoices"])
if not df_inv.empty:
    df_inv = df_inv[["status", "amount_usd"] + [c for c in df_inv.columns
                                                  if c not in ("status", "amount_usd")]]
    print(f"Last {len(df_inv)} invoices:")
    df_inv
else:
    print("No invoices yet — they appear after your first paid API call.")


---
## Ticker Snapshot — Latest Risk Metrics

`/api/metrics/{ticker}` returns a single-row snapshot of the most recent risk metrics for any ticker: volatility, Sharpe ratio, all hedge ratios, factor explained returns, sector codes, market cap, and close price. Useful for a quick dashboard or screening table.


In [ ]:
# ── Ticker metrics snapshot ─────────────────────────────────────────────────────
# Single ticker — returns all hedge ratios (hr), explained-risk fractions (er),
# volatility, Sharpe, market cap, and sector codes in one call.
ticker = "NVDA"
resp = requests.get(f"{BASE_URL}/metrics/{ticker}", headers=HEADERS)
resp.raise_for_status()
m = resp.json()

# ── Transpose for readability ─────────────────────────────────────────────────
rows = {
    # Price & size
    "date":                   m.get("date"),
    "close_price":            m.get("close_price"),
    "market_cap_B":           round(m["market_cap"] / 1e9, 1) if m.get("market_cap") else None,
    # Risk / return
    "volatility (ann %)":     round(m["volatility"] * 100, 2) if m.get("volatility") else None,
    "sharpe_ratio":           round(m["sharpe_ratio"], 4) if m.get("sharpe_ratio") else None,
    # Hedge ratios — one SPY ratio per level, plus sector/subsector
    "l1_market_hr":           m.get("l1_market_hr"),
    "l2_market_hr":           m.get("l2_market_hr"),
    "l2_sector_hr":           m.get("l2_sector_hr"),
    "l3_market_hr":           m.get("l3_market_hr"),
    "l3_sector_hr":           m.get("l3_sector_hr"),
    "l3_subsector_hr":        m.get("l3_subsector_hr"),
    # Factor explained-risk fractions (0–1; sum of all four ≈ 1)
    "l1_market_er":           m.get("l1_market_er"),
    "l2_market_er":           m.get("l2_market_er"),
    "l2_sector_er":           m.get("l2_sector_er"),
    "l3_market_er":           m.get("l3_market_er"),
    "l3_sector_er":           m.get("l3_sector_er"),
    "l3_subsector_er":        m.get("l3_subsector_er"),
    "l3_residual_er":         m.get("l3_residual_er"),   # idiosyncratic — cannot be hedged
    # Sector codes
    "bw_sector_code":         m.get("bw_sector_code"),
    "fs_sector_code":         m.get("fs_sector_code"),
    "fs_industry_code":       m.get("fs_industry_code"),
}

df_snap = pd.DataFrame.from_dict(rows, orient="index", columns=["Value"])
print(f"{ticker}  —  full risk metrics snapshot")
display(df_snap)


### Bonus: Metrics screening table across a list of tickers


In [ ]:
# ── Metrics screening table ─────────────────────────────────────────────────────
tickers = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "JPM", "TSLA"]

rows = []
for t in tickers:
    r = requests.get(f"{BASE_URL}/metrics/{t}", headers=HEADERS)
    if r.status_code == 200:
        rows.append(r.json())
    else:
        print(f"Warning: {t} returned {r.status_code}")

df_screen = pd.DataFrame(rows)
display_cols = [
    "ticker", "date", "close_price", "market_cap", "volatility", "sharpe_ratio",
    "l1_market_hr", "l2_market_hr", "l2_sector_hr",
    "l3_market_hr", "l3_sector_hr", "l3_subsector_hr",
    "bw_sector_code",
]
df_screen[[c for c in display_cols if c in df_screen.columns]]
